In [1]:
import pandas as pd
import numpy as np


In [6]:
from pathlib import Path

p = Path('data/processed/train_data.parquet')

if p.exists():
    df = pd.read_parquet(p)
else:
    alternatives = [
        Path('data/processed/train.parquet'),
        Path('data/train_data.parquet'),
        Path('../data/processed/train_data.parquet'),
    ]
    found = next((a for a in alternatives if a.exists()), None)
    if found:
        if found.suffix == '.parquet':
            df = pd.read_parquet(found)
            
    else:
        proc = list(Path('data/processed').iterdir()) if Path('data/processed').exists() else []
        data_root = list(Path('data').iterdir()) if Path('data').exists() else []
        raise FileNotFoundError(
            f"train_data.parquet not found at {p!s}. Checked alternatives: {[str(a) for a in alternatives]}. "
            f"Files in data/processed: {[str(x) for x in proc]}. Files in data: {[str(x) for x in data_root]}."
        )

In [18]:
df.columns

Index(['hours_streaming', 'hours_social', 'hours_messaging', 'hours_gaming',
       'is_peak_hour_user', 'is_weekend', 'age_group', 'plan_type',
       'device_type', 'network_type', 'streaming_data_gb', 'social_data_gb',
       'messaging_data_gb', 'gaming_data_gb', 'total_data_gb', 'day_of_week',
       'month'],
      dtype='object')

In [10]:
remove_cols =['user_id','data_usage_category',  'total_usage_gb', 'top_activity', 'day_of_week', 'hour',
       'churn_risk_score', 'arpu_zar', 'arpu_per_gb']

df = df.drop(columns=remove_cols)


In [28]:
df.columns

Index(['hours_streaming', 'hours_social', 'hours_messaging', 'hours_gaming',
       'is_peak_hour_user', 'is_weekend', 'age_group', 'plan_type',
       'device_type', 'network_type', 'streaming_data_gb', 'social_data_gb',
       'messaging_data_gb', 'gaming_data_gb', 'total_data_gb', 'day_of_week',
       'month'],
      dtype='object')

In [13]:
df['day_of_week'] =df['measurement_date'].dt.dayofweek
df['month'] = df['measurement_date'].dt.month

df = df.drop(columns=['measurement_date'])


In [ ]:
# Based on the current dataframe, we should apply:
# - ordinal encoding for age_group, plan_type, and network_type
# - nominal encoding for device_type
# - cyclical encoding for day_of_week and month 


from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# 1. ORDINAL ENCODING 
age_group_order = [['18-24', '25-34', '35-44', '45-54', '55+']]
plan_type_order = [['Prepaid_Daily', 'Prepaid_Monthly', 'Postpaid_Basic', 'Postpaid_Premium', 'Postpaid_Unlimited']]
network_type_order = [['3G', '4G', '4G+', '5G']] 

# Apply ordinal encoding
ordinal_encoder_age = OrdinalEncoder(categories=age_group_order)
ordinal_encoder_plan = OrdinalEncoder(categories=plan_type_order)
ordinal_encoder_network = OrdinalEncoder(categories=network_type_order)

df['age_group_encoded'] = ordinal_encoder_age.fit_transform(df[['age_group']])
df['plan_type_encoded'] = ordinal_encoder_plan.fit_transform(df[['plan_type']])
df['network_type_encoded'] = ordinal_encoder_network.fit_transform(df[['network_type']])

# Drop original ordinal columns 
df = df.drop(['age_group', 'plan_type', 'network_type'], axis=1)


# 2. NOMINAL ENCODING for device_type
device_dummies = pd.get_dummies(df['device_type'], prefix='device', drop_first=True)
df = pd.concat([df, device_dummies], axis=1)
df = df.drop('device_type', axis=1)



# 3. CYCLICAL ENCODING for day_of_week and month
# Day of week (1-7, where 1=Monday, 7=Sunday)
if df['day_of_week'].dtype == 'object':
    day_mapping = {
        'Monday': 1, 'Tuesday': 2, 'Wednesday': 3, 'Thursday': 4,
        'Friday': 5, 'Saturday': 6, 'Sunday': 7
    }
    df['day_of_week_num'] = df['day_of_week'].map(day_mapping)
else:
    df['day_of_week_num'] = df['day_of_week']

# Month (1-12)
if df['month'].dtype == 'object':
    month_mapping = {
        'January': 1, 'February': 2, 'March': 3, 'April': 4,
        'May': 5, 'June': 6, 'July': 7, 'August': 8,
        'September': 9, 'October': 10, 'November': 11, 'December': 12
    }
    df['month_num'] = df['month'].map(month_mapping)
else:
    df['month_num'] = df['month']

# Apply cyclical encoding using sine and cosine transformations
# For day of week (period=7)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week_num'] / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week_num'] / 7)

# For month (period=12)
df['month_sin'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month_num'] / 12)

# Drop intermediate numeric columns
df = df.drop(['day_of_week_num', 'month_num'], axis=1)
# Drop original columns if they exist
if 'day_of_week' in df.columns:
    df = df.drop('day_of_week', axis=1)
if 'month' in df.columns:
    df = df.drop('month', axis=1)

# View the result
print("Encoded dataframe shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nColumn names:")
print(df.columns.tolist())

Encoded dataframe shape: (8004, 22)

First few rows:
   hours_streaming  hours_social  hours_messaging  hours_gaming  \
0             1.27          6.00             0.15          0.00   
1             0.33          0.22             1.79          0.31   
2             0.54          1.57             0.39          0.10   
3             0.24          1.31             1.03          0.65   
4             1.90          0.68             0.01          0.22   

   is_peak_hour_user  is_weekend  streaming_data_gb  social_data_gb  \
0                  1           0            2.28387         1.83908   
1                  0           0            0.12846         0.03291   
2                  0           0            0.37170         0.19520   
3                  0           1            0.20155         0.11429   
4                  0           0            4.30271         0.19975   

   messaging_data_gb  gaming_data_gb  ...  plan_type_encoded  \
0            0.00208         0.00000  ...            

In [31]:
df.head()

,hours_streaming,hours_social,hours_messaging,hours_gaming,is_peak_hour_user,is_weekend,age_group,plan_type,device_type,network_type,streaming_data_gb,social_data_gb,messaging_data_gb,gaming_data_gb,total_data_gb,day_of_week,month,age_group_encoded,plan_type_encoded,network_type_encoded
0,1.27,6.00,0.15,0.00,1,0,35-44,Prepaid_Daily,Premium_Smartphone,5G,2.28387,1.83908,0.00208,0.00000,4.45799,4,3,2.0,0.0,3.0
1,0.33,0.22,1.79,0.31,0,0,25-34,Postpaid_Basic,Premium_Smartphone,3G,0.12846,0.03291,0.02319,0.01750,0.24945,0,3,1.0,2.0,0.0
2,0.54,1.57,0.39,0.10,0,0,25-34,Postpaid_Unlimited,Mid_Range,3G,0.37170,0.19520,0.00540,0.01295,0.61941,5,4,1.0,4.0,0.0
3,0.24,1.31,1.03,0.65,0,1,18-24,Postpaid_Premium,Mid_Range,3G,0.20155,0.11429,0.01589,0.08245,0.40725,6,4,0.0,3.0,0.0
4,1.90,0.68,0.01,0.22,0,0,18-24,Postpaid_Basic,Basic_Phone,5G,4.30271,0.19975,0.00012,0.00968,4.66772,4,4,0.0,2.0,3.0
